In [77]:
from __future__ import annotations
from dataclasses import dataclass
from types import MappingProxyType
from typing import Mapping, Sequence

import numpy as np
from numpy.typing import NDArray

FloatVector = NDArray[np.float64]

# Objectives

We want to represent out state vector in a central and standard repeatable format. The idea is that I want to provide a way to reference different states easily but leave the door open for the expansion to different state vector models. 

For instance, if we have a Longitudinal 3-DoF aircraft model we have the states:

- X:     Position North
- Z:     Position Down
- u:     Body Velocity along x-axis
- w:     Body Velocity along z-axis
- q:     Angular velocity, pitch rate
- $\theta$:      Pitch Angle


We can represent this as a flat numpy vector but when we expand to 6-DoF or augmented state vector representations then the rest of our program should not have to know what index each element is directly but should be able to call for the x_position and out comes the position north from the state vector. To acomplish this we can use slices and map the slices to each.

In [78]:

def validate_vector(values: FloatVector | Sequence[float], expected_size: int, vector_name: str) -> FloatVector:
    array = np.asarray(values, dtype=np.float64)

    if array.ndim != 1:
        raise ValueError( f"{vector_name} must be 1-D, recived {array.shape}.")

    if array.size != expected_size:
        raise ValueError(f"{vector_name} must contain {expected_size} values, recived {array.size}")

    if not np.all(np.isfinite(array)):
        raise ValueError(f"{vector_name} contains NaN or Infinite values.")

    result = array.copy()
    result.setflags(write=False)

    return result


@dataclass(frozen=True, slots=True)
class StateLayout:
    slices: Mapping[str, slice]
    size: int

    @classmethod
    def from_fields(cls, fields: Sequence[tuple[str, int]]) -> StateLayout:

        slices: dict[str, slice] = {}
        start = 0

        for name, width in fields:
            if name in slices:
                raise ValueError(f"Duplicate State Field: {name}")

            if width <= 0:
                raise ValueError(f"State Field '{name}' must have positive width.")

            slices[name] = slice(start, start + width)
            start += width

        return cls(slices=MappingProxyType(slices), size=start)

    def get_slice(self, name: str) -> slice:
        try:
            return self.slices[name]
        except KeyError as exc:
            raise KeyError(f"Unknown state field: {name}") from exc


@dataclass(frozen=True, slots=True)
class VehicleState:
    layout: StateLayout
    values: FloatVector

    def __post_init__(self) -> None:
        validated = validate_vector(self.values, expected_size=self.layout.size, vector_name="VehicleState")
        object.__setattr__(self, "values", validated)

    def section(self, name: str) -> FloatVector:
        return self.values[self.layout.get_slice(name)]

    def scalar(self, name: str) -> float:
        values = self.section(name)

        if values.size != 1:
            raise ValueError(f" State Field '{name}' constains {values.size} values, not one.")

        return float(values[0])

    def with_values(self, values: FloatVector | Sequence[float]) -> VehicleState:
        return VehicleState(layout=self.layout, values = np.asarray(values, dtype=np.float64))


@dataclass(frozen=True, slots=True)
class StateDerivative:
    layout: StateLayout
    values: FloatVector

    def __post_init__(self) -> None:
        validated = validate_vector(self.values, expected_size=self.layout.size, vector_name="StateDerivative")
        object.__setattr__(self, "values", validated)


In [79]:
LONGITUDINAL_LAYOUT = StateLayout.from_fields(
    [
        ("position_ned_ft", 2),     # north, down
        ("velocity_body_fps", 2),   # u, w
        ("pitch_rate_rad_s", 1),    # q
        ("pitch_angle_rad", 1)      # theta
    ]
)

initial_state = VehicleState(
    layout=LONGITUDINAL_LAYOUT,
    values=np.array(
        [
            0.0,       # north position
            -1000.0,   # down position; -1000 ft means 1000 ft altitude
            80.0,      # u
            0.0,       # w
            0.0,       # q
            0.0,       # theta
        ],
        dtype=np.float64,
    ),
)

In [80]:
position = initial_state.section("position_ned_ft")
velocity = initial_state.section("velocity_body_fps")

pitch_rate = initial_state.scalar("pitch_rate_rad_s")
pitch_angle = initial_state.scalar("pitch_angle_rad")

print("=============================================")
print("Initial State Check")
print(" ")
print(f"    Position:       {position} ft")
print(f"    Velocity:       {velocity} ft/s")
print(f"    Pitch Angle:    {pitch_rate} rad")
print(f"    Pitch Rate:     {pitch_angle} rad/s")
print(" ")
print("=============================================")


Initial State Check
 
    Position:       [    0. -1000.] ft
    Velocity:       [80.  0.] ft/s
    Pitch Angle:    0.0 rad
    Pitch Rate:     0.0 rad/s
 


In [81]:
@dataclass(frozen=True, slots=True)
class MassProperties:
    mass_slug: float
    inertia_body_slug_ft2: NDArray[np.float64]
    cg_body_ft: NDArray[np.float64]

    def __post_init__(self) -> None:
        inertia = np.asarray(self.inertia_body_slug_ft2, dtype=np.float64)
        cg = np.asarray(self.cg_body_ft, dtype=np.float64)

        if inertia.shape != (3,3):
            raise ValueError("Inertia Matrix must have shape (3, 3).")

        if cg.shape != (3,):
            raise ValueError("Center of Gravity must have shaper (3,).")

        if self.mass_slug <= 0.0:
            raise ValueError("Mass must be positive.")

        object.__setattr__(self, "inertia_body_slug_ft2", inertia.copy())
        object.__setattr__(self, "cg_body_ft", cg.copy())



In [82]:
from typing import Callable, Protocol
from dataclasses import dataclass

StateEquation = Callable[[float, VehicleState], StateDerivative]

StateProjector = Callable[[VehicleState], VehicleState]

class Integrator(Protocol):
    def step(self, equation: StateEquation, time_s: float, state: VehicleState, step_size_s: float) -> VehicleState:
        ...


@dataclass
class ForwardEulerIntegrator:
    projector: StateProjector | None = None

    def step(self, equation: StateEquation, time_s: float, state: VehicleState, step_size_s: float) -> VehicleState:

        if step_size_s <= 0.0:
            raise ValueError("Integration step must be positive.")

        derivitive = equation(time_s, state)

        self._verify_layout(state, derivitive)

        next_values = state.values + step_size_s * derivitive.values

        next_state = state.with_values(next_values)

        if self.projector is not None:
            next_state = self.projector(next_state)

        return next_state



    @staticmethod
    def _verify_layout(state: VehicleState, derivitive: StateDerivative) -> None:
        if derivitive.layout != state.layout:
            raise ValueError("StateDerivitive layout does not match VehicleState Layout.")
    






In [83]:
from dataclasses import dataclass
import math

@dataclass
class LongitudinalAircraftDynamics:
    mass_properties: MassProperties
    #atmosphere_model: object
    #aerodynamic_model: object
    #propulsion_model: object
    #actuator_model: object

    def derivatives(
        self,
        time_s: float,
        state: VehicleState,
    ) -> StateDerivative:
        altitude_ft = -state.section("position_ned_ft")[1]

        x       = state.section("position_ned_ft")[0]
        z       = state.section("position_ned_ft")[1]
        u       = state.section("velocity_body_fps")[0]
        w       = state.section("velocity_body_fps")[1]
        q       = state.scalar("pitch_rate_rad_s")
        pitch   = state.scalar("pitch_angle_rad")

        mass = self.mass_properties.mass_slug
        Iyy = self.mass_properties.inertia_body_slug_ft2[1,1]

        g = 32.17

        #atmosphere = self.atmosphere_model.evaluate(altitude_ft)

        # Calculate wind and air data.

        # Calculate actuator outputs.

        # Calculate propulsion forces.

        # Calculate aerodynamic forces and moments.
        Fx = -mass * g * math.sin(pitch)
        Fz = mass * g * math.cos(pitch)
        M  = 0.0

        # Calculate rigid-body derivatives.

        u_dot = -q * w + Fx / mass
        w_dot = q * u + Fz / mass
        q_dot = M / Iyy
        x_dot = u * math.cos(pitch) + w * math.sin(pitch)
        z_dot = -u * math.sin(pitch) + w * math.cos(pitch)
        pitch_dot = q

        derivative_values = np.array(
            [x_dot, z_dot, u_dot, w_dot, q_dot, pitch_dot],
            dtype=np.float64,
        )

        return StateDerivative(
            layout=state.layout,
            values=derivative_values,
        )

In [84]:
mass_properties = MassProperties(mass_slug=100.0, 
                                 inertia_body_slug_ft2=np.array([[1.0, 0.0, 0.0], [0.0, 1.0, 0.0], [0.0, 0.0, 1.0]]), 
                                 cg_body_ft=np.array([1.0, 0.0, 0.0]))

dynamics_model = LongitudinalAircraftDynamics(mass_properties)
integrator = ForwardEulerIntegrator()
next_state = integrator.step(equation=dynamics_model.derivatives, time_s = 0.0, state=initial_state, step_size_s=0.01)


position = next_state.section("position_ned_ft")
velocity = next_state.section("velocity_body_fps")

pitch_rate = next_state.scalar("pitch_rate_rad_s")
pitch_angle = next_state.scalar("pitch_angle_rad")

print("=============================================")
print("Integrated State Check")
print(" ")
print(f"    Position:       {position} ft")
print(f"    Velocity:       {velocity} ft/s")
print(f"    Pitch Angle:    {pitch_rate} rad")
print(f"    Pitch Rate:     {pitch_angle} rad/s")
print(" ")
print("=============================================")

Integrated State Check
 
    Position:       [ 8.e-01 -1.e+03] ft
    Velocity:       [80.      0.3217] ft/s
    Pitch Angle:    0.0 rad
    Pitch Rate:     0.0 rad/s
 
